# Expo Memory Synth: بناء المشهد 3D من فيديو

هذا النوتبوك ياخذ فيديو مصور بالجوال، ويطلع منه ملف Gaussian Splatting (`.ply`) تعرضونه بالموقع. يشتغل على **Google Colab** و **Kaggle**.

**الخطوات:** فيديو ← فريمات ← (طمس الوجيه) ← COLMAP يحسب أماكن الكاميرا ← تدريب gsplat ← ملف `.ply`

**قبل ما تبدين:**
- **Colab:** من القائمة `Runtime > Change runtime type` واختاري **T4 GPU**.
- **Kaggle:** من اللوحة اليمين `Settings`: خلي **Accelerator** على **GPU T4 x2**، وشغلي **Internet** (يحتاج توثيق الحساب برقم الجوال مرة وحدة).
- شغلي الخلايا **وحدة وحدة بالترتيب**، لا تستخدمين `Run all`.
- خلي الجهاز على الشاحن وما ينام (Sleep)، لأن الجلسة تنقطع إذا نام الجهاز.
- **إذا خلصت حصة الـ GPU بـ Colab** (رسالة `Cannot connect to GPU backend`): اختاري `Connect without GPU` وسوي الخطوات لين COLMAP (ما يحتاج GPU)، والنتيجة تنحفظ بالـ Drive. لما ترجع الحصة استخدمي خلية **الاسترجاع** قبل التدريب. أو كملي على Kaggle.

## 1. تأكدي إن فيه GPU
لازم يطلع **Tesla T4**. إذا بتسوين COLMAP بس بدون GPU، عادي تطلع هنا رسالة خطأ.

In [ ]:
!nvidia-smi -L

## 2. الإعدادات والفيديو
- `VIDEO_NAME`: اسم الفيديو.
- `FILE_ID`: رقم الملف من رابط Google Drive (الجزء بعد `/d/` أو `id=`). الملف لازم يكون مشارك "أي شخص معه الرابط".
  - **Kaggle:** لازم تعبينه.
  - **Colab:** إما تعبينه، أو تحطين الفيديو داخل `My Drive/expo-memory-synth` وتتركينه فاضي.
- `SCENE_NAME`: اسم مجلد النتايج. غيريه لكل مشهد (مثلاً `rock` ، `hall`) عشان النتايج ما تختلط.

In [ ]:
import os, subprocess
ON_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ
print('Platform:', 'Kaggle' if ON_KAGGLE else 'Colab')

VIDEO_NAME = 'IMG_0920.MOV'   # <-- اسم الفيديو
FILE_ID    = '1QYyPC6s2JLeJh-8_6pJt5-rrJ6AUw223'   # <-- رقم الملف من رابط Drive (هذا IMG_0920)
SCENE_NAME = 'hall'           # <-- اسم المشهد

if ON_KAGGLE:
    WORK = '/kaggle/working'                      # ينحفظ مع النوتبوك، ويتنزل من لوحة Output
    OUT  = f'{WORK}/outputs/{SCENE_NAME}'
    VIDEO = f'/tmp/{VIDEO_NAME}'
else:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT = '/content/drive/MyDrive/expo-memory-synth'
    WORK = '/content'
    OUT  = f'{PROJECT}/outputs/{SCENE_NAME}'     # بالـ Drive عشان ما تضيع لو انقطعت الجلسة
    VIDEO = f'{PROJECT}/{VIDEO_NAME}'

SCENE = f'{WORK}/{SCENE_NAME}'
os.makedirs(f'{SCENE}/images', exist_ok=True)
os.makedirs(OUT, exist_ok=True)

if FILE_ID:
    VIDEO = f'/tmp/{VIDEO_NAME}'
    if not os.path.exists(VIDEO):
        !pip -q install gdown
        !gdown {FILE_ID} -O "{VIDEO}"

ok = os.path.exists(VIDEO)
print('Video exists:', ok, f'({os.path.getsize(VIDEO)/1e6:.0f} MB)' if ok else '<- عبي FILE_ID أو شيكي اسم الفيديو')

## 3. تثبيت الأدوات (3-5 دقايق)
- **ffmpeg** لتقطيع الفيديو، و**COLMAP** لحساب أماكن الكاميرا، و**gsplat** للتدريب (إصدار 1.5.3).
- الأخطاء الحمراء عن `numpy` بآخر التثبيت طبيعية، المهم يطلع `COLMAP` و `done`.
- **إذا طلب Restart session:** اختاري **Restart session** بس (مو Disconnect and delete runtime)، وبعدين شغلي خلية 2 من جديد وكملي من خلية 4.
- بدون GPU تتثبت أدوات COLMAP بس.

In [ ]:
import torch
HAS_GPU = torch.cuda.is_available()
GSPLAT_VERSION = '1.5.3'
!apt-get -qq update && apt-get -qq install -y ffmpeg colmap > /dev/null
if HAS_GPU:
    !pip -q install gsplat=={GSPLAT_VERSION}
    !rm -rf {WORK}/gsplat
    !git clone -q --depth 1 --branch v{GSPLAT_VERSION} https://github.com/nerfstudio-project/gsplat.git {WORK}/gsplat
    !pip -q install -r {WORK}/gsplat/examples/requirements.txt
    # examples/datasets has no __init__.py, so the preinstalled `datasets` package would shadow it
    !touch {WORK}/gsplat/examples/datasets/__init__.py
    # gsplat pins numpy<2, which breaks the preinstalled TensorFlow; the trainer only needs tensorboard
    !pip -q uninstall -y tensorflow
else:
    print('ما فيه GPU: ثبتت أدوات COLMAP بس. التدريب يحتاج جلسة GPU.')
!colmap -h | head -1
print('done')

## 4. تقطيع الفيديو لفريمات
النوتبوك يحسب عدد الصور بالثانية تلقائياً عشان يطلع قريب من `TARGET_FRAMES`:
- **قطعة وحدة** (فيديو 30-60 ثانية): `TARGET_FRAMES = 180`
- **جناح كامل** (فيديو 3-6 دقايق): `TARGET_FRAMES = 350`. أكثر من كذا COLMAP يطول ساعات.

إذا تبين جزء معين من الفيديو بس، عبي `START` و `END`.

In [ ]:
TARGET_FRAMES = 350
MAX_FPS = 3
WIDTH   = 1600
START = ''   # مثال: '00:00:10'  (اتركيه فاضي للفيديو كامل)
END   = ''   # مثال: '00:05:30'

def seconds(t):
    h, m, s = ([0, 0] + [float(x) for x in t.split(':')])[-3:]
    return h * 3600 + m * 60 + s

probe = subprocess.run(['ffprobe', '-v', 'error', '-show_entries', 'format=duration', '-of', 'csv=p=0', VIDEO],
                       capture_output=True, text=True).stdout.strip()
duration = float(probe or 0)
if END:   duration = seconds(END)
if START: duration -= seconds(START)
FPS = round(min(MAX_FPS, TARGET_FRAMES / duration), 3) if duration > 0 else MAX_FPS
print(f'Video: {duration:.0f} s -> {FPS} frames per second')

trim = (f'-ss {START} ' if START else '') + (f'-to {END} ' if END else '')
!rm -f {SCENE}/images/*.jpg
!ffmpeg -hide_banner -loglevel error {trim} -i "{VIDEO}" -vf "fps={FPS},scale='min({WIDTH},iw)':-2" -q:v 2 {SCENE}/images/frame_%04d.jpg

n = len(os.listdir(f'{SCENE}/images'))
print(f'{n} frames')
if n < 60:  print('قليل! زيدي TARGET_FRAMES أو استخدمي مقطع أطول')

### (اختياري) شوفي عينة من الصور
تأكدي إنها مو مقلوبة ولا مغبشة. إذا ألوانها باهتة مرة، غالباً الفيديو مصور بـ HDR.

In [ ]:
from IPython.display import Image, display
import glob
frames = sorted(glob.glob(f'{SCENE}/images/*.jpg'))
for f in frames[::max(1, len(frames) // 4)][:4]:
    display(Image(f, width=300))

## 5. (اختياري) طمس الوجيه للخصوصية
إذا طلع خطأ بـ `cv2` أو `numpy` تخطيها، ما تأثر على المشهد.

In [ ]:
import cv2, glob
det = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
total = 0
for f in sorted(glob.glob(f'{SCENE}/images/*.jpg')):
    img = cv2.imread(f)
    faces = det.detectMultiScale(cv2.cvtColor(img, cv2.COLOR_BGR2GRAY), 1.1, 5, minSize=(30, 30))
    for (x, y, w, h) in faces:
        img[y:y+h, x:x+w] = cv2.GaussianBlur(img[y:y+h, x:x+w], (51, 51), 0)
    if len(faces):
        cv2.imwrite(f, img); total += len(faces)
print('faces blurred:', total)

## 6. COLMAP: حساب مكان الكاميرا لكل صورة
هذي أهم خطوة، وتشتغل على المعالج (CPU) مو الـ GPU، فطبيعي Colab يقول إنك ما تستخدمين الـ GPU. سكري التنبيه بس.

- **قطعة وحدة (~180 صورة):** 15-30 دقيقة.
- **جناح كامل (~350 صورة):** 40-90 دقيقة.

للفيديوهات الطويلة النوتبوك يشغل **loop detection**: يتعرف لما ترجعين لمكان صورتيه قبل، ويربطه. هذا يمنع المشهد يتقسم أو يلتوي.

In [ ]:
os.environ['QT_QPA_PLATFORM'] = 'offscreen'   # no display on Colab/Kaggle
n = len(os.listdir(f'{SCENE}/images'))

!rm -rf {SCENE}/database.db {SCENE}/sparse {SCENE}/sparse_parts && mkdir -p {SCENE}/sparse

!colmap feature_extractor --database_path {SCENE}/database.db --image_path {SCENE}/images \
    --ImageReader.single_camera 1 --ImageReader.camera_model OPENCV --SiftExtraction.use_gpu 0

match = ['colmap', 'sequential_matcher', '--database_path', f'{SCENE}/database.db',
         '--SiftMatching.use_gpu', '0', '--SequentialMatching.overlap', '15']
loop = []
if n > 200:
    VOCAB = '/tmp/vocab_tree_flickr100K_words32K.bin'
    if not os.path.exists(VOCAB):
        !wget -q -O {VOCAB} https://github.com/colmap/colmap/releases/download/3.11.1/vocab_tree_flickr100K_words32K.bin
    if os.path.exists(VOCAB) and os.path.getsize(VOCAB) > 1e6:
        loop = ['--SequentialMatching.loop_detection', '1', '--SequentialMatching.vocab_tree_path', VOCAB]
print('loop detection:', bool(loop))
if subprocess.run(match + loop).returncode != 0 and loop:
    print('loop detection ما اشتغل، أعيد المطابقة بدونه')
    subprocess.run(match, check=True)

!colmap mapper --database_path {SCENE}/database.db --image_path {SCENE}/images --output_path {SCENE}/sparse

### تحققي من النتيجة
الخلية تختار أكبر جزء طلع (لو COLMAP قسم المشهد) وتحفظ نسخة احتياطية.
- **Registered قريب من عدد الصور:** ممتاز.
- **أقل من 70%:** غالباً فيه حركة سريعة أو أماكن فاضية بالتصوير. المشهد بيطلع ناقص بس تقدرين تكملين.

In [ ]:
import re, shutil
def analyze(path):
    out = subprocess.run(['colmap', 'model_analyzer', '--path', path], capture_output=True, text=True)
    text = out.stdout + out.stderr
    m = re.search(r'Registered images:\s*(\d+)', text)
    return (int(m.group(1)) if m else 0), text

models = sorted(d for d in os.listdir(f'{SCENE}/sparse') if os.path.isdir(f'{SCENE}/sparse/{d}'))
stats = {m: analyze(f'{SCENE}/sparse/{m}') for m in models}
for m, (k, _) in stats.items():
    print(f'part {m}: {k} / {n} images')
best = max(stats, key=lambda m: stats[m][0])
if best != '0':
    # gsplat reads sparse/0, so the largest part goes there
    os.makedirs(f'{SCENE}/sparse_parts', exist_ok=True)
    shutil.move(f'{SCENE}/sparse/0', f'{SCENE}/sparse_parts/0')
    shutil.move(f'{SCENE}/sparse/{best}', f'{SCENE}/sparse/0')
k, text = stats[best]
print('\n'.join(l for l in text.splitlines() if re.search('Cameras|Images|Registered|Points|reprojection', l)))
if k < 0.7 * n:
    print(f'\nانبنى {k} من {n} صورة بس. المشهد بيطلع ناقص.')

if not ON_KAGGLE:
    !rm -rf "{OUT}/sparse" "{OUT}/images"
    !cp -r {SCENE}/sparse {SCENE}/images "{OUT}/"   # نسخة احتياطية بالـ Drive
    print('Backup saved to Drive:', OUT)

### (Colab بس) استرجاع نتيجة COLMAP من الـ Drive
إذا انقطعت الجلسة بعد COLMAP، أو سويتي COLMAP بدون GPU ورجعتي الحين بجلسة GPU: شغلي خلية 1 و 2 و 3، وبعدين هذي الخلية، وروحي للتدريب مباشرة.

In [ ]:
import shutil
if not os.path.exists(f'{SCENE}/sparse/0'):
    shutil.rmtree(SCENE, ignore_errors=True)
    shutil.copytree(f'{OUT}/images', f'{SCENE}/images')
    shutil.copytree(f'{OUT}/sparse', f'{SCENE}/sparse')
print('frames:', len(os.listdir(f'{SCENE}/images')), '| sparse:', os.listdir(f'{SCENE}/sparse'))

## 7. تدريب الـ Gaussian Splatting (يحتاج GPU)
- **قطعة وحدة:** `MAX_STEPS = 7000`، حوالي 15-30 دقيقة على T4.
- **جناح كامل:** `MAX_STEPS = 15000` أو أكثر، حوالي 40-60 دقيقة.
- أول مرة ممكن يقعد 5-10 دقايق بدون ما يطلع شي (يسوي compile لأكواد CUDA)، هذا طبيعي.
- إذا طلع خطأ `out of memory`: غيري `DATA_FACTOR` لـ `2` (يصغر الصور للنص).

In [ ]:
import torch
assert torch.cuda.is_available(), 'ما فيه GPU: التدريب يحتاج T4'
MAX_STEPS   = 15000
DATA_FACTOR = 1
RESULTS = f'{WORK}/results/{SCENE_NAME}'

%cd {WORK}/gsplat/examples
!python simple_trainer.py default \
    --data_dir {SCENE} --data_factor {DATA_FACTOR} --result_dir {RESULTS} \
    --max_steps {MAX_STEPS} --save_ply --ply_steps {MAX_STEPS} --disable_viewer
%cd {WORK}

## 8. حفظ ملف `.ply` وتنزيله

In [ ]:
import glob, shutil
plys = sorted(glob.glob(f'{RESULTS}/**/*.ply', recursive=True), key=os.path.getmtime)
if plys:
    final = f'{OUT}/{SCENE_NAME}.ply'
    shutil.copy(plys[-1], final)
    print('Saved:', final, f'({os.path.getsize(final)/1e6:.1f} MB)')
    if ON_KAGGLE:
        from IPython.display import FileLink, display
        display(FileLink(os.path.relpath(final, WORK)))
        print('اضغطي الرابط فوق، أو نزليه من لوحة Output على اليمين.')
    else:
        print('نزليه من Drive: My Drive > expo-memory-synth > outputs >', SCENE_NAME)
else:
    print('ما لقيت ملف ply. شيكي مخرجات خلية التدريب.')

## 9. الخطوة الجاية
1. افتحي الملف بـ **SuperSplat** (https://superspl.at/editor)، احذفي النقاط الطايرة حول الأطراف، وصدريه بـ `File > Export > Compressed PLY` (يصغر الحجم كثير).
2. حطيه بالريبو داخل `web/` باسم `scene.ply` وشغلي الموقع (التفاصيل بالـ README).
3. حطي النقاط التفاعلية من `index.html?edit`.